# Systems and Phase Planes

A system $u' = F(u)$ with two variables can be drawn in the **phase plane**: each point is a state and each trajectory a history. For linear systems $u' = Au$ the eigenvalues of $A$ decide everything. For nonlinear systems, the eigenvalues of the Jacobian at an equilibrium decide what happens nearby.

In [ ]:
Pollis.packages("OrdinaryDiffEq", "ForwardDiff", "Plots")

In [ ]:
using OrdinaryDiffEq, LinearAlgebra, ForwardDiff, Plots

## Four linear systems

| Eigenvalues | Type |
|---|---|
| Real, both negative | Stable node |
| Real, opposite signs | Saddle |
| Complex, negative real part | Stable spiral |
| Purely imaginary | Centre |

Equivalently: stable when the trace is negative and the determinant positive; oscillating when $\text{tr}^2 < 4\det$.

In [ ]:
cases = ["stable node" => [-2.0 0.0; 0.0 -0.5], "saddle" => [1.0 0.0; 0.0 -1.0],
         "stable spiral" => [-0.3 -2.0; 2.0 -0.3], "centre" => [0.0 -1.0; 1.0 0.0]]
[(case = name, eigenvalues = eigvals(A), trace = tr(A), determinant = det(A)) for (name, A) in cases]

In [ ]:
function portrait(A, title)
    xs = range(-2, 2, length=15)
    X = vec([x for x in xs, y in xs]); Y = vec([y for x in xs, y in xs])
    D = [A * [x, y] for (x, y) in zip(X, Y)]
    len = [norm(v) + eps() for v in D]
    p = quiver(X, Y, quiver=(0.2 .* first.(D) ./ len, 0.2 .* last.(D) ./ len), c=:gray, title=title, xlims=(-2.2, 2.2), ylims=(-2.2, 2.2), aspect_ratio=1)
    for u0 in ([1.8, 0.3], [-1.5, 1.5], [0.2, -1.8], [-1.0, -1.0])
        plot!(p, solve(ODEProblem((u, p, t) -> A * u, u0, (0.0, 6.0)), Tsit5()), idxs=(1, 2), lw=2, label="")
    end
    return p
end
plot([portrait(A, name) for (name, A) in cases]..., layout=(2, 2), size=(800, 800))

The exact solution of $u' = Au$ is $u(t) = e^{At}u_0$, where `exp` of a matrix is the matrix exponential. The solver reproduces it:

In [ ]:
A = cases[3][2]; u0 = [1.0, 0.0]
sol = solve(ODEProblem((u, p, t) -> A * u, u0, (0.0, 5.0)), Tsit5(); reltol=1e-10, abstol=1e-12)
norm(sol(5.0) - exp(5.0 * A) * u0)

## Predator and prey

Lotka and Volterra: prey $x$ grow at rate $a$ and are eaten at rate $bxy$; predators $y$ die at rate $c$ and grow at rate $dxy$ by eating.

$$x' = ax - bxy, \qquad y' = dxy - cy.$$

These are the default equations of the System of Equations tab in the panel.

In [ ]:
a, b, c, d = 1.1, 0.4, 0.4, 0.1
lv(u, p, t) = [a * u[1] - b * u[1] * u[2], d * u[1] * u[2] - c * u[2]]
sol = solve(ODEProblem(lv, [10.0, 5.0], (0.0, 50.0)), Tsit5(); reltol=1e-10, abstol=1e-12)
plot(plot(sol, lw=2, label=["prey" "predators"], xlabel="t"),
     plot(sol, idxs=(1, 2), lw=2, xlabel="prey", ylabel="predators", label=""), layout=(1, 2), size=(900, 350))

The coexistence equilibrium is $(c/d,\ a/b)$. Its Jacobian has purely imaginary eigenvalues $\pm i\sqrt{ac}$, a centre, so small cycles have period about $2\pi/\sqrt{ac}$.

In [ ]:
ueq = [c / d, a / b]
J = ForwardDiff.jacobian(u -> lv(u, nothing, 0.0), ueq)
(equilibrium = ueq, eigenvalues = eigvals(J), small_cycle_period = 2pi / sqrt(a * c))

For a centre, the linearisation cannot say whether cycles persist. Here they do. The quantity $V = dx - c\ln x + by - a\ln y$ is constant along every solution, so trajectories are closed curves. A good solver keeps $V$ constant to near the tolerance.

In [ ]:
V(u) = d * u[1] - c * log(u[1]) + b * u[2] - a * log(u[2])
(V_min = minimum(V.(sol.u)), V_max = maximum(V.(sol.u)))

## Linearising a nonlinear system

Two species competing for the same resource:

$$x' = x(1 - x - 0.5y), \qquad y' = y(0.75 - y - 0.25x).$$

There are four equilibria: extinction, $x$ alone, $y$ alone, and coexistence, where both brackets vanish. The Jacobian's eigenvalues classify each.

In [ ]:
comp(u) = [u[1] * (1 - u[1] - 0.5u[2]), u[2] * (0.75 - u[2] - 0.25u[1])]
eqs = [[0.0, 0.0], [1.0, 0.0], [0.0, 0.75], [1.0 0.5; 0.25 1.0] \ [1.0, 0.75]]
[(equilibrium = e, residual = norm(comp(e)), eigenvalues = eigvals(ForwardDiff.jacobian(comp, e))) for e in eqs]

In [ ]:
p = plot(xlabel="x", ylabel="y", title="Competition: every mixed start ends at coexistence", xlims=(0, 1.3), ylims=(0, 1.0))
for u0 in ([0.05, 0.05], [1.2, 0.9], [0.1, 0.9], [1.2, 0.05], [0.6, 0.02])
    plot!(p, solve(ODEProblem((u, p, t) -> comp(u), u0, (0.0, 60.0)), Tsit5()), idxs=(1, 2), lw=2, label="")
end
scatter!(p, first.(eqs), last.(eqs), ms=6, c=:black, label="equilibria")

## Exercises

1. Make the predators more efficient ($d = 0.2$). Where does the equilibrium move, and does the period of small cycles change?
2. Add logistic growth for the prey, $ax(1 - x/20)$. Classify the coexistence equilibrium now.
3. In the competition model, change 0.5 to 1.5 and 0.25 to 1.5. Which equilibria are stable, and what does that mean ecologically?
4. Find a $2 \times 2$ matrix with trace $-1$ and determinant 2. Predict the type of equilibrium, then check with `portrait`.